# Core systems and diagrams

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alx87grd/minilink/blob/main/examples/notebooks/intro/00_core.ipynb)

Official API intro to minilink's continuous-time core: `System` / `DynamicSystem`,
ports, writing `f` / `h` (then the same models with `params`), diagram operators (`+`, `>>`, `@`), manual wiring, and
simulation façades.

**Scripts for depth:** `examples/scripts/diagrams/`

**See also:** [`showcase/minilink.ipynb`](../showcase/minilink.ipynb) · [`intro/01_blocks.ipynb`](01_blocks.ipynb) · [`intro/05_simulation.ipynb`](05_simulation.ipynb)


In [ ]:
# Local conda: minilink already installed. Colab: clone + path + meshcat.
import sys

if "google.colab" in sys.modules:
    get_ipython().run_line_magic("matplotlib", "inline")
    get_ipython().system("git clone https://github.com/alx87grd/minilink")
    sys.path.insert(0, "/content/minilink")
    get_ipython().system("pip install -q meshcat")


## 1. Quick start

**Pass 1 — what a `System` is**, using catalog defaults only (no `params` dict yet).
One plant: visualize **f**, simulate, then close the loop.


### A plant by itself

Every minilink model is a **`System`**: named input/output **ports** and a state
vector `x`. The contract is one **dynamics** equation plus one map **per output
port**:

    dx_dt = f(x, u, t)          # how the state evolves
    y     = h(x, u, t)          # one equation per output port

Catalog plants ship with sensible defaults — construct, set `x0` if you want,
and display the object.


In [ ]:
import numpy as np
from minilink.dynamics.catalog.pendulum.pendulum import Pendulum

sys = Pendulum()
sys


Visualize **f**: `plot_phase_plane()` draws the
vector field of the dynamics — how the state would move at each point.


In [ ]:
sys.plot_phase_plane()

In [ ]:
sys.x0[0] = 2.0
sys.compute_trajectory(tf=10.0)
sys.plot_trajectory()
sys.plot_phase_plane()

### Closed loop: `controller @ plant`

The `@` operator builds the standard feedback loop. Chain a step reference with
`>>` into the controller input.


In [ ]:
from minilink.blocks.sources import Step
from minilink.control.impedance import ImpedanceController

step = Step(final_value=np.array([2.0]), step_time=5.0)
ctl = ImpedanceController()  # default gains
diagram = step >> ctl @ sys
diagram.compute_trajectory(tf=10.0)
diagram.plot_trajectory()


## 2. Everything is a System

A `System` owns its equations (`f`, `h`), named **ports**, and initial condition.
Trajectories live in the returned `Trajectory` — not hidden inside the object.


In [ ]:
print("Plant:   ", sys.name)
print("states:  ", sys.n, sys.state.labels, sys.state.units)
print("inputs:  ", sys.m, list(sys.inputs))
print("outputs: ", list(sys.outputs))
print("Diagram: ", diagram.n, sys.state.labels, sys.state.units)


## 3. Write your own model

Still **pass 1**: subclass `DynamicSystem` and write $\dot x = f(x,u,t)$ like the
textbook — coefficients as plain locals, no parameter plumbing. `output_dim=2`
creates the standard `y` port so `>>` can wire the diagram.


In [ ]:
from minilink.core.system import DynamicSystem


class MassSpringDamper(DynamicSystem):
    # m x'' + c x' + k x = u

    def __init__(self):
        super().__init__(n=2, input_dim=1, output_dim=2)

    def f(self, x, u, t=0, params=None):
        # m \ddot{p} + c \dot{p} + k p = f
        m = 1.0  # mass [kg]
        k = 4.0  # stiffness [N/m]
        c = 0.3  # damping [N.s/m]
        p = x[0]  # position [m]
        v = x[1]  # velocity [m/s]
        f = u[0]  # applied force [N]
        a = (f - c * v - k * p) / m  # acceleration [m/s^2]
        return np.array([v, a])


msd = MassSpringDamper()
msd.x0[0] = 1.0
msd_chain = Step(final_value=np.array([10.0]), step_time=2.0) >> msd
msd_chain.compute_trajectory(tf=20.0, n_steps=1001)
msd_chain.plot_trajectory()


## 4. Parameters

**Pass 2 — the `params` system.** Coefficients for reproducible tuning (and later
autodiff w.r.t. physical constants) live in `self.params` and are read from
`f`'s `params` argument.

Contract: `params is None` means “use the object defaults (`self.params`)”; any
other dict overrides. See [`showcase/jax.ipynb`](jax.ipynb) for ∂f/∂params.

### Custom plant (same mass–spring–damper ODE as §3)

Same $\dot x = f$, but `m`, `k`, `c` come from `params`. Sources use
`step.params[...]` the same way:


In [ ]:
class MassSpringDamperParams(DynamicSystem):
    # m \ddot{p} + c \dot{p} + k p = f  (same ODE as §3)

    def __init__(self):
        super().__init__(n=2, input_dim=1, output_dim=2)
        self.params = {"m": 1.0, "k": 4.0, "c": 0.3}

    def f(self, x, u, t=0, params=None):
        if params is None:
            params = self.params
        m = params["m"]  # mass [kg]
        k = params["k"]  # stiffness [N/m]
        c = params["c"]  # damping [N.s/m]
        p = x[0]  # position [m]
        v = x[1]  # velocity [m/s]
        f = u[0]  # applied force [N]
        a = (f - c * v - k * p) / m  # acceleration [m/s^2]
        return np.array([v, a])


msd_p = MassSpringDamperParams()
msd_p.params["c"] = 0.8  # retune damping without rewriting f
msd_p.x0[0] = 1.0

step_msd = Step()
step_msd.params["initial_value"] = np.array([0.0])
step_msd.params["final_value"] = np.array([10.0])
step_msd.params["step_time"] = 2.0

msd_p_chain = step_msd >> msd_p
msd_p_chain.compute_trajectory(tf=20.0, n_steps=1001)
msd_p_chain.plot_trajectory()


### Catalog diagram (same wiring idea, catalog blocks)

A closed loop from catalog plants/controllers — tune each block through
`.params`:


In [ ]:
plant = Pendulum()
plant.params["l"] = 1.2  # arm length [m]
plant.params["d"] = 0.3  # damping [N.s/m]
plant.x0[0] = 2.0  # initial angle [rad]

step = Step()
step.params["initial_value"] = np.array([0.0])
step.params["final_value"] = np.array([2.0])
step.params["step_time"] = 5.0

ctl = ImpedanceController()
ctl.params["Kp"] = 100.0
ctl.params["Kd"] = 10.0

loop = step >> ctl @ plant
loop.compute_trajectory(tf=10.0)
loop.plot_trajectory()


## 5. Diagram operators


In [ ]:
auto = (Step() + ImpedanceController() + Pendulum()).autowire(strict=True)
auto.plot_diagram()

## 6. Manual diagram wiring


In [ ]:
from minilink.core.diagram import DiagramSystem

sys = Pendulum()
sys.params["m"] = 1.0
sys.params["l"] = 5.0
sys.x0[0] = 2.0

step = Step()
step.params["initial_value"] = np.array([0.0])
step.params["final_value"] = np.array([1.0])
step.params["step_time"] = 10.0

ctl = ImpedanceController()
ctl.params["Kp"] = 1000.0
ctl.params["Kd"] = 100.0

diagram = DiagramSystem()
diagram.add_subsystem(step, "step")
diagram.add_subsystem(ctl, "controller")
diagram.add_subsystem(sys, "plant")

diagram.connect("step", "y", "controller", "r")
diagram.connect("controller", "u", "plant", "u")
diagram.connect("plant", "y", "controller", "y")

diagram.plot_diagram()
diagram.compute_trajectory(tf=20)
diagram.plot_trajectory()


In [ ]:
# step.show_signal(t0=0.0, tf=20.0)

## 7. Custom systems and cascade


In [ ]:
from minilink.core.diagram import DiagramSystem
from minilink.core.system import System


class Integrator(DynamicSystem):
    def __init__(self):
        super().__init__(n=1, input_dim=1, output_dim=1, y_dependencies=())
        self.name = "Int"

    def f(self, x, u, t=0, params=None):
        dx = np.zeros(self.n)
        dx[0] = u[0]
        return dx

    def h(self, x, u, t=0, params=None):
        y = np.zeros(self.p)
        y[0] = x[0]
        return y


class PropController(System):
    def __init__(self):
        super().__init__()
        self.params = {"Kp": 10.0}
        self.name = "Controller"
        self.add_input_port("r", nominal_value=0.0)
        self.add_input_port("y", nominal_value=0.0)
        self.add_output_port("u", function=self.ctl, dependencies=("r", "y"))

    def ctl(self, x, u, t=0, params=None):
        if params is None:
            params = self.params
        r, y = self.get_port_values_from_u(u, "r", "y")
        return np.array([params["Kp"] * (r[0] - y[0])])


sys1 = Integrator()
sys1.state.labels = ["v"]
sys1.x0[0] = 2.0
sys2 = Integrator()
sys2.state.labels = ["x"]
sys2.x0[0] = 2.0

ctl1 = PropController()
ctl1.params["Kp"] = 1.0
ctl2 = PropController()
ctl2.params["Kp"] = 1.0

step = Step()
step.params["initial_value"] = np.array([0.0])
step.params["final_value"] = np.array([1.0])
step.params["step_time"] = 10.0

cascade = DiagramSystem()
cascade.add_subsystem(step, "step")
cascade.add_subsystem(ctl1, "controller1")
cascade.add_subsystem(ctl2, "controller2")
cascade.add_subsystem(sys1, "integrator1")
cascade.add_subsystem(sys2, "integrator2")

cascade.connect("integrator1", "y", "integrator2", "u")
cascade.connect("controller2", "u", "integrator1", "u")
cascade.connect("integrator1", "y", "controller2", "y")
cascade.connect("controller1", "u", "controller2", "r")
cascade.connect("integrator2", "y", "controller1", "y")
cascade.connect("step", "y", "controller1", "r")

cascade


In [ ]:
cascade.compute_trajectory(tf=20)
cascade.plot_trajectory()

## Noise ports

Disturbance / noise channels are ordinary ports — wire them like any other signal.


In [ ]:
from minilink.blocks.sources import WhiteNoise
from minilink.core.diagram import DiagramSystem
from minilink.dynamics.catalog.pendulum.pendulum import PendulumWithNoisePort

sys = PendulumWithNoisePort()
sys.params["m"] = 1.0
sys.params["l"] = 5.0
sys.x0[0] = 2.0

step = Step()
step.params["initial_value"] = np.array([0.0])
step.params["final_value"] = np.array([1.0])
step.params["step_time"] = 10.0

noise = WhiteNoise(1)
noise.params.update({"var": 1.0, "mean": 0.0, "seed": 1})
noise2 = WhiteNoise(1)
noise2.params.update({"var": 0.1, "mean": 0.0, "seed": 2})

ctl = ImpedanceController()
ctl.params["Kp"] = 1000.0
ctl.params["Kd"] = 100.0

diagram2 = DiagramSystem()
diagram2.add_subsystem(step, "step")
diagram2.add_subsystem(ctl, "controller")
diagram2.add_subsystem(sys, "plant")
diagram2.add_subsystem(noise, "noise")
diagram2.add_subsystem(noise2, "noise2")

diagram2.connect("step", "y", "controller", "r")
diagram2.connect("controller", "u", "plant", "u")
diagram2.connect("plant", "y", "controller", "y")
diagram2.connect("noise", "y", "plant", "w")
diagram2.connect("noise2", "y", "plant", "v")

diagram2


In [ ]:
diagram2.compute_trajectory(solver="euler", dt=0.01)
diagram2.plot_trajectory()

In [ ]:
noise2.show_signal(t0=-300.0, tf=300.0)